# Notebook 01: トークン化と埋め込み (Tokenization & Embedding)

LLM はテキストをそのまま処理できません。
**テキスト → 整数列 → 実数ベクトル列** という変換が必要です。

このノートでは次の流れを実際の数値で確認します：

```
"hello" → [7, 4, 11, 11, 14] → embedding vectors → + positional encoding → X
```

**使うライブラリ**：numpy のみ（ディープラーニングフレームワーク不要）

In [1]:
import numpy as np
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

---
## 1. 文字レベルのトークン化

最もシンプルな「1文字 = 1トークン」方式で仕組みを理解します。

実際の LLM（GPT-4 など）は BPE（後述）を使い、単語の一部をトークンにしますが、
計算の流れは同じです。

In [2]:
# 学習テキスト（小さいサンプル）
text = "hello world"

# 語彙（vocabulary）を構築：テキスト中のユニーク文字をソート
vocab = sorted(set(text))
print(f"語彙: {vocab}")
print(f"語彙サイズ: {len(vocab)}")

語彙: [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
語彙サイズ: 8


In [3]:
# 文字 ↔ 整数 の対応辞書を作る
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

print("char → idx:", char2idx)
print()

# テキストをトークン ID 列に変換
token_ids = [char2idx[c] for c in text]
print(f"'{text}' → {token_ids}")
print(f"各文字: {list(zip(text, token_ids))}")

char → idx: {' ': 0, 'd': 1, 'e': 2, 'h': 3, 'l': 4, 'o': 5, 'r': 6, 'w': 7}

'hello world' → [3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1]
各文字: [('h', 3), ('e', 2), ('l', 4), ('l', 4), ('o', 5), (' ', 0), ('w', 7), ('o', 5), ('r', 6), ('l', 4), ('d', 1)]


In [4]:
# 逆変換：整数列 → テキスト
decoded = "".join(idx2char[i] for i in token_ids)
print(f"デコード: {token_ids} → '{decoded}'")

デコード: [3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1] → 'hello world'


---
## 2. BPE（Byte Pair Encoding）の直感的理解

GPT などで使われる BPE は「よく出るペアを 1 つのトークンにまとめる」アルゴリズムです。

### 手順
1. 文字単位からスタート
2. 最も頻出するペアを探す
3. そのペアを新しいトークンとして語彙に追加
4. テキストを更新（ペアを新トークンで置換）
5. 語彙サイズが目標に達するまで繰り返す

小さい例で実際に動かしてみます。

In [5]:
def get_pair_counts(tokens):
    """隣り合うペアの出現回数を数える"""
    counts = {}
    for pair in zip(tokens[:-1], tokens[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge_pair(tokens, pair, new_token):
    """tokens 中の pair をすべて new_token に置換"""
    result = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
            result.append(new_token)
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return result

# 例：'a'=0, 'b'=1, 'c'=2 として "aabcaab" を BPE
tokens_bpe = [0, 0, 1, 2, 0, 0, 1]  # a a b c a a b
idx2tok = {0: 'a', 1: 'b', 2: 'c'}
next_id = 3

print(f"初期: {[idx2tok[t] for t in tokens_bpe]}")
print()

for step in range(2):  # 2 回マージ
    counts = get_pair_counts(tokens_bpe)
    best_pair = max(counts, key=counts.get)
    new_name = idx2tok[best_pair[0]] + idx2tok[best_pair[1]]
    idx2tok[next_id] = new_name
    print(f"Step {step+1}: 最頻ペア {counts}")
    print(f"  → '{idx2tok[best_pair[0]]}'+'{idx2tok[best_pair[1]]}' を '{new_name}'(id={next_id}) にマージ")
    tokens_bpe = merge_pair(tokens_bpe, best_pair, next_id)
    print(f"  → トークン列: {[idx2tok[t] for t in tokens_bpe]}")
    print()
    next_id += 1

初期: ['a', 'a', 'b', 'c', 'a', 'a', 'b']

Step 1: 最頻ペア {(0, 0): 2, (0, 1): 2, (1, 2): 1, (2, 0): 1}
  → 'a'+'a' を 'aa'(id=3) にマージ
  → トークン列: ['aa', 'b', 'c', 'aa', 'b']

Step 2: 最頻ペア {(3, 1): 2, (1, 2): 1, (2, 3): 1}
  → 'aa'+'b' を 'aab'(id=4) にマージ
  → トークン列: ['aab', 'c', 'aab']



---
## 3. Embedding テーブル（埋め込み行列）

整数のトークン ID を「意味のある実数ベクトル」に変換します。

**Embedding テーブル**は `(vocab_size, d_model)` の行列です。
トークン ID = 行番号として、対応する行ベクトルを取り出すだけです。

$$\text{embed}(i) = E[i, :]$$

この重みは学習によって調整されますが、最初はランダム初期化します。

In [6]:
# 設定（小さい値で全数値が見えるように）
vocab_size = 8   # 語彙サイズ
d_model    = 4   # 埋め込み次元（実際の LLM では 768〜12288）

# Embedding テーブル：各トークンに対応する d_model 次元のベクトル
E = np.random.randn(vocab_size, d_model)

print(f"Embedding テーブル E: shape = {E.shape}")
print(f"（行 = トークン ID, 列 = 特徴次元）")
print()
print("E =\n", E)

Embedding テーブル E: shape = (8, 4)
（行 = トークン ID, 列 = 特徴次元）

E =
 [[ 0.4967 -0.1383  0.6477  1.523 ]
 [-0.2342 -0.2341  1.5792  0.7674]
 [-0.4695  0.5426 -0.4634 -0.4657]
 [ 0.242  -1.9133 -1.7249 -0.5623]
 [-1.0128  0.3142 -0.908  -1.4123]
 [ 1.4656 -0.2258  0.0675 -1.4247]
 [-0.5444  0.1109 -1.151   0.3757]
 [-0.6006 -0.2917 -0.6017  1.8523]]


In [7]:
# トークン ID から埋め込みベクトルを取り出す
token_id = 3
vec = E[token_id]
print(f"token_id={token_id} の埋め込みベクトル: {vec}")
print(f"shape: {vec.shape}")

print()
# これは「行ベクトルを取り出す」だけ
print(f"E[3, :] = {E[3, :]}  ← 同じ")

token_id=3 の埋め込みベクトル: [ 0.242  -1.9133 -1.7249 -0.5623]
shape: (4,)

E[3, :] = [ 0.242  -1.9133 -1.7249 -0.5623]  ← 同じ


In [8]:
# 複数トークンの埋め込みを一度に取得
# NumPy のインデックス配列で一行操作
tokens = [2, 5, 1, 3]   # 「hello」の最初の4文字に対応するID（仮）
X_embed = E[tokens]     # shape: (seq_len, d_model) = (4, 4)

print(f"トークン列: {tokens}")
print(f"埋め込み行列 X_embed: shape = {X_embed.shape}")
print()
print("X_embed =\n", X_embed)

トークン列: [2, 5, 1, 3]
埋め込み行列 X_embed: shape = (4, 4)

X_embed =
 [[-0.4695  0.5426 -0.4634 -0.4657]
 [ 1.4656 -0.2258  0.0675 -1.4247]
 [-0.2342 -0.2341  1.5792  0.7674]
 [ 0.242  -1.9133 -1.7249 -0.5623]]


---
## 4. Positional Encoding（位置符号化）

Embedding だけでは「どのトークンが何番目か」の情報がありません。
Self-Attention は並列処理なので、順序を別途与える必要があります。

**Transformer 論文（Vaswani 2017）の正弦波式**：

$$
\text{PE}(pos, 2i)   = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$
$$
\text{PE}(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

- 偶数次元 → sin, 奇数次元 → cos
- 次元によって周波数が異なる → 各位置がユニークなベクトル

In [9]:
def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            angle = pos / (10000 ** (2 * i / d_model))
            PE[pos, i]   = np.sin(angle)
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(angle)
    return PE

seq_len = 4
PE = positional_encoding(seq_len, d_model)

print(f"Positional Encoding PE: shape = {PE.shape}")
print(f"（行 = トークン位置, 列 = 次元）")
print()
print("PE =\n", PE)
print()
print("→ 各行（位置）でベクトルが異なることを確認")

Positional Encoding PE: shape = (4, 4)
（行 = トークン位置, 列 = 次元）

PE =
 [[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.0001  1.    ]
 [ 0.9093 -0.4161  0.0002  1.    ]
 [ 0.1411 -0.99    0.0003  1.    ]]

→ 各行（位置）でベクトルが異なることを確認


In [10]:
# 各次元の周波数を確認
print("次元ごとの周波数（低次元 = 低周波, 高次元 = 高周波）:")
for i in range(0, d_model, 2):
    freq = 1.0 / (10000 ** (2*i / d_model))
    print(f"  次元 {i},{i+1}: 周波数 = {freq:.4f}")

次元ごとの周波数（低次元 = 低周波, 高次元 = 高周波）:
  次元 0,1: 周波数 = 1.0000
  次元 2,3: 周波数 = 0.0001


---
## 5. 入力行列 X = Embedding + Positional Encoding

Transformer への最終的な入力はこの和です：

$$X = \text{Embed}(\text{tokens}) + \text{PE}$$

形状は `(seq_len, d_model)` — これが以降の全ステップへの入力になります。

In [11]:
X = X_embed + PE   # 要素ごとの足し算 (4,4) + (4,4) = (4,4)

print("=" * 50)
print(f"X_embed（埋め込み）:")
print(X_embed)
print()
print(f"PE（位置符号化）:")
print(PE)
print()
print(f"X = X_embed + PE（Transformer への入力）: shape = {X.shape}")
print(X)
print("=" * 50)
print()
print("この X が Notebook 02（Self-Attention）への入力になります")

X_embed（埋め込み）:
[[-0.4695  0.5426 -0.4634 -0.4657]
 [ 1.4656 -0.2258  0.0675 -1.4247]
 [-0.2342 -0.2341  1.5792  0.7674]
 [ 0.242  -1.9133 -1.7249 -0.5623]]

PE（位置符号化）:
[[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.0001  1.    ]
 [ 0.9093 -0.4161  0.0002  1.    ]
 [ 0.1411 -0.99    0.0003  1.    ]]

X = X_embed + PE（Transformer への入力）: shape = (4, 4)
[[-0.4695  1.5426 -0.4634  0.5343]
 [ 2.3071  0.3145  0.0676 -0.4247]
 [ 0.6751 -0.6503  1.5794  1.7674]
 [ 0.3831 -2.9033 -1.7246  0.4377]]

この X が Notebook 02（Self-Attention）への入力になります


---
## まとめ

| ステップ | 入力 | 出力 | 形状 |
|---------|------|------|------|
| トークン化 | "hello" | [7,4,11,11,14] | `(seq_len,)` |
| Embedding | [7,4,11,11,14] | 浮動小数点行列 | `(seq_len, d_model)` |
| Positional Encoding | 位置インデックス | sin/cos 行列 | `(seq_len, d_model)` |
| **X（最終入力）** | **Embed + PE** | **Transformer入力** | **`(seq_len, d_model)`** |

→ **次のノートブック `02_self_attention.ipynb`** では、この X から Q/K/V を作り Attention を計算します。